# Enter the finite calculus

Start with three yes/no results for one two-stage trace: the first local check, the second local check, and the end-to-end check. This lesson computes their mismatch, then shows the exact hypothesis needed to sum over an intermediate boundary.

![Finite defect diagnostics](../public_export/figure_lggr_phase_boundary.png)

## One trace, one defect

For local verdicts $E_1,E_2$ and global verdict $E_g$, set

$$L=E_1E_2, \qquad \Delta_E=E_g-L.$$

The value $-1$ is the local-green/global-red case: both stages pass, but the complete trace fails. The five-row fixture below is exact finite data, not a population estimate.

In [ ]:
from epk_nb import import_epk, widgets_available
import matplotlib.pyplot as plt

epk = import_epk()
HAS_WIDGETS = widgets_available()
sample = epk.make_five_trace_phone_test()
report = epk.EPKMetrics.compute(sample)
summary = {
    'LPR': report.lpr,
    'GPR': report.gpr,
    'mean Delta_E': report.mean_delta_e,
    'NGD': report.ngd_all,
    'EGD': report.egd,
}
print(summary)
assert summary == {'LPR': 0.6, 'GPR': 0.2, 'mean Delta_E': -0.4, 'NGD': 0.4, 'EGD': 0.4}
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(list(summary), list(summary.values()), color=['#4c78a8', '#f58518', '#e45756', '#b279a2', '#72b7b2'])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylim(-0.5, 0.7)
ax.set_ylabel('finite sample value')
ax.set_title('Five-trace diagnostic fixture')
plt.show()

## What exact composition assumes

G1 is a two-sided correspondence, not merely a way to split paths already observed:

$$\Omega(a,c) \cong \bigsqcup_{b\in\mathsf B} \bigl(\Omega(b,c)\times\Omega(a,b)\bigr).$$

Every global path has one compatible segment pair, and every compatible pair produces one global path. With multiplicative weights (G2) and factored evaluators (G3), the evaluated kernel composes. Without G3, the exact residual is

$$Z_E(c\mid a)=\sum_{b\in\mathsf B}Z_{E_2}(c\mid b)Z_{E_1}(b\mid a)+R_E(c,a).$$

In [ ]:
a, b, c = map(epk.State, ['a', 'b', 'c'])
left = [epk.TrajectorySegment(a, b, ('draft',)), epk.TrajectorySegment(a, b, ('revise',))]
right = [epk.TrajectorySegment(b, c, ('check',))]
global_paths = [epk.ComposedTrajectory(l, r) for l in left for r in right]
valid, violations = epk.BoundarySlice({b}).validate_unique_decomposition(
    global_paths, left_segments=left, right_segments=right
)
print({'G1': valid, 'global paths': len(global_paths), 'compatible pairs': len(left) * len(right)})
assert valid and not violations

## Change the end-to-end verdict

The control changes only $E_g$. Notice that local evidence stays fixed while the defect changes. In a headless run the same calculation uses $E_g=0$.

In [ ]:
def show_defect(global_pass: bool) -> None:
    trace = epk.EvaluatorVerdictTriple(1, 1, int(global_pass))
    print({'local product': trace.local_product, 'global verdict': trace.eg, 'Delta_E': trace.delta_eval})

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    display(widgets.interactive(show_defect, global_pass=widgets.Checkbox(value=False, description='global pass')))
else:
    show_defect(False)

## Checkpoint

Change one row of the fixture from $(1,1,0)$ to $(1,1,1)$ and predict LPR, GPR, NGD, and the signed mean before rerunning it. Then read manuscript Sections 2–4. The matching Lean entry points are `B1_local_green_global_red_witness`, `C1c_g1_set_isomorphism`, and `D2_defect_corrected_kernel_identity`. The computations illustrate finite records; they do not establish a benchmark result.